<a href="https://colab.research.google.com/github/niran-j2005/assesment/blob/main/case_study_Niranjan_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
url = "https://raw.githubusercontent.com/erkansirin78/datasets/master/AB_NYC_2019.csv"
df = pd.read_csv(url)
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


Part 1 — Data Understanding & Quality Audit

In [6]:
# TODO: shape, dtypes, missing value summary
print("Shape of dataset:", df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Value Summary:")
print(df.isnull().sum())

Shape of dataset: (48895, 16)

Data Types:
id                                  int64
name                               object
host_id                             int64
host_name                          object
neighbourhood_group                object
neighbourhood                      object
latitude                          float64
longitude                         float64
room_type                          object
price                               int64
minimum_nights                      int64
number_of_reviews                   int64
last_review                        object
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
dtype: object

Missing Value Summary:
id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitud

In [7]:
#check for values that are technically present but don't make real-world sense
print("Zero/negative price:", (df["price"] <= 0).sum())
print("Minimum nights > 1095:", (df["minimum_nights"] > 1095).sum())
print("Availability > 365:", (df["availability_365"] > 365).sum())
print("Negative reviews:", (df["number_of_reviews"] < 0).sum())

Zero/negative price: 11
Minimum nights > 1095: 1
Availability > 365: 0
Negative reviews: 0


In [8]:
#  duplicate check


print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 0


**Written notes — what did you find, and what looks suspicious?**

Findings: The dataset contains 48,895 rows and 16 columns. Missing values were found in name, host_name, last_review, and reviews_per_month. Most of the missing values are in last_review and reviews_per_month, with 10,052 missing values each.

Suspicious values: 11 listings have a price of zero or less, which is unrealistic for a nightly rental price. One listing has a minimum stay of more than 1,095 days, which is unusually high for a short-term rental. No invalid availability values or negative review counts were found. No exact duplicate rows were found.

Part 2 — Missing Value Diagnosis & Treatment

In [9]:
#  investigate missingness pattern(s) across columns

print("Missing values by column:")
print(df.isnull().sum())

print("\nCheck if missing last_review is related to zero reviews:")
print(pd.crosstab(df["number_of_reviews"] == 0,
                  df["last_review"].isnull()))

print("\nCheck if missing reviews_per_month is related to zero reviews:")
print(pd.crosstab(df["number_of_reviews"] == 0,
                  df["reviews_per_month"].isnull()))

Missing values by column:
id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

Check if missing last_review is related to zero reviews:
last_review        False  True 
number_of_reviews              
False              38843      0
True                   0  10052

Check if missing reviews_per_month is related to zero reviews:
reviews_per_month  False  True 
number_of_reviews              
False  

**Written justification — MCAR / MAR / MNAR classification per column, and your reasoning:**

name – Approximately MCAR
Only 16 values are missing, and there is no obvious pattern in the available data.

host_name – Approximately MCAR
Only 21 values are missing, and there is no obvious pattern in the available data.

last_review – MAR
All 10,052 missing values occur when number_of_reviews = 0.

reviews_per_month – MAR
All 10,052 missing values occur when number_of_reviews = 0.

Conclusion: name and host_name appear approximately MCAR, while last_review and reviews_per_month are MAR because their missingness is explained by the observed number_of_reviews column. No strong evidence of MNAR was found.Your answer here._

In [10]:
# apply your chosen treatment(s)
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

# Drop columns that are not needed for the model
df = df.drop(columns=["name", "host_name", "last_review"])

print("Missing values after treatment:")
print(df.isnull().sum())

Missing values after treatment:
id                                0
host_id                           0
neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64


**Written justification — why this treatment for each column, and what you'd risk with `dropna()` instead:**

Treatment and Reasoning

name – Dropped
Only 16 values are missing, and the listing name is not needed for our basic price prediction model.

host_name – Dropped
Only 21 values are missing, and the host's name is not a useful feature for predicting price.

last_review – Dropped
It is missing when a listing has no reviews, so there is no date to fill in. It also represents past activity that may not be available when predicting the price of a new listing.

reviews_per_month – Filled with 0
When number_of_reviews = 0, there are no reviews per month. Therefore, replacing the missing value with 0 is logically appropriate.

Using dropna() would remove every row containing a missing value. This would unnecessarily remove valid listings, especially the 10,052 listings with no reviews, reducing the dataset and potentially introducing bias._Your answer here._

Part 3 — Outlier Detection & Treatment

In [11]:

def find_outliers(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return df[(df[column] < lower) | (df[column] > upper)]

# Outliers in price
price_outliers = find_outliers("price")
print("Price outliers:", len(price_outliers))

# Outliers in minimum_nights
nights_outliers = find_outliers("minimum_nights")
print("Minimum nights outliers:", len(nights_outliers))


Price outliers: 2972
Minimum nights outliers: 6607


**Written justification — for each outlier group, is it an error or a genuine listing? What evidence supports your call?**

Outlier Assessment: The IQR method identified 2,972 price outliers and 6,607 minimum-night outliers. These values are not automatically considered errors. Many high-price listings can be genuine, especially for entire homes or apartments in Manhattan. Similarly, some listings can legitimately have higher minimum-stay requirements. However, the 11 listings with price less than or equal to 0 are considered invalid, and the one listing with a minimum stay above 1,095 days is considered unrealistic and should be removed._Your answer here._

In [12]:
#  apply treatment consistent with your judgment above
df = df[df["price"] > 0].copy()

# Remove unrealistic minimum-night value
df = df[df["minimum_nights"] <= 1095].copy()

print("New shape:", df.shape)

New shape: (48883, 13)


Part 4 — Feature Engineering & Encoding

In [13]:
#  encode categorical columns appropriately
categorical_columns = [
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

print(df.head())

     id  host_id  latitude  longitude  price  minimum_nights  \
0  2539     2787  40.64749  -73.97237    149               1   
1  2595     2845  40.75362  -73.98377    225               1   
2  3647     4632  40.80902  -73.94190    150               3   
3  3831     4869  40.68514  -73.95976     89               1   
4  5022     7192  40.79851  -73.94399     80              10   

   number_of_reviews  reviews_per_month  calculated_host_listings_count  \
0                  9               0.21                               6   
1                 45               0.38                               2   
2                  0               0.00                               1   
3                270               4.64                               1   
4                  9               0.10                               1   

   availability_365  ...  neighbourhood_Williamsbridge  \
0               365  ...                         False   
1               355  ...                        

In [14]:
# 1. engineer at least two new features
import numpy as np
# 1. Distance from the center of Manhattan
df["distance_to_manhattan"] = np.sqrt(
    (df["latitude"] - 40.7831)**2 +
    (df["longitude"] + 73.9712)**2
)

# 2. Log of minimum nights
df["minimum_nights_log"] = np.log1p(df["minimum_nights"])

print(df[[
    "distance_to_manhattan",
    "minimum_nights_log"
]].head())

   distance_to_manhattan  minimum_nights_log
0               0.135615            0.693147
1               0.032048            0.693147
2               0.039120            1.386294
3               0.098626            0.693147
4               0.031271            2.397895


**Written justification — why these features, and which column(s) should NOT be used to predict price, and why:**

Feature Selection: The engineered feature distance_to_manhattan was selected because location is an important factor that can affect Airbnb rental prices. minimum_nights_log was selected because it reduces the effect of very large minimum-stay values and may help the model capture pricing patterns more effectively.

Columns Not Used for Prediction: The id and host_id columns should not be used because they are identifiers and do not describe the actual property. name and host_name are also excluded because they are not reliable general features for predicting price. last_review, number_of_reviews, and reviews_per_month are excluded because they describe past activity and may not be available when predicting the price of a new listing, which could lead to data leakage._Your answer here._

Part 5 — Build a Reusable Preprocessing Pipeline

In [15]:
%pip install scikit-learn

In [16]:
#  train/test split (before fitting anything)

from sklearn.model_selection import train_test_split

X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (39106, 237)
Testing data: (9777, 237)


In [17]:
# T build Pipeline combining your cleaning, imputation, encoding, scaling steps


import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Reload original dataset
data = pd.read_csv(url)

# Remove invalid values
data = data[data["price"] > 0]
data = data[data["minimum_nights"] <= 1095]

# Create new features
data["distance_to_manhattan"] = np.sqrt(
    (data["latitude"] - 40.7831)**2 +
    (data["longitude"] + 73.9712)**2
)

data["minimum_nights_log"] = np.log1p(data["minimum_nights"])

# Remove columns not used for prediction
data = data.drop(columns=[
    "id", "host_id", "name", "host_name",
    "last_review", "number_of_reviews", "reviews_per_month"
])

# Separate features and target
X = data.drop(columns=["price"])
y = data["price"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Column lists
numeric_features = [
    "latitude",
    "longitude",
    "minimum_nights",
    "availability_365",
    "calculated_host_listings_count",
    "distance_to_manhattan",
    "minimum_nights_log"
]

categorical_features = [
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

# Numeric pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Final preprocessing pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor)
])

# Fit only on training data
pipeline.fit(X_train)

print("Pipeline created successfully!")

Pipeline created successfully!


---
## Part 6 — Written Reflection

1. Which preprocessing decision had the largest effect on your model's performance?

The treatment of extreme values in price was one of the most important preprocessing decisions. Removing invalid prices and handling the highly skewed price distribution helps the model make more reliable predictions. This can be checked by comparing the model's evaluation scores before and after preprocessing.
_Your answer._


2. If StayScope Analytics started receiving live listing data, which part of your pipeline would need to change, and why?

The data validation and preprocessing stage would need to handle new incoming listings continuously. It should check for new missing values, invalid values, and changes in the data distribution while ensuring that only information available at prediction time is used._Your answer._


3. Why didn't you just delete every row with a missing or unusual value?

Not every missing or unusual value is an error. Deleting all such rows could remove valid listings, reduce the amount of useful data, and introduce bias into the model._Your answer.